# Phase 1 — Exploration & Chargement des données
**Data Challenge Agriculture — Togo AI Lab (Défi 2)**

Les fichiers CSV ont été téléchargés manuellement et déposés dans `data/raw/`.
Ce notebook les charge, les explore et prépare les GeoDataFrames pour la Phase 2.

In [21]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..')
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

# Vérification : liste les fichiers présents dans data/raw/
print('Fichiers dans data/raw/ :')
for f in sorted(RAW.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} Ko)')

Fichiers dans data/raw/ :
  abattoirs.csv  (7.8 Ko)
  agri_rural.csv  (136.5 Ko)
  barrages.csv  (115.3 Ko)
  elevage.csv  (1447.6 Ko)
  pisciculture.csv  (62.4 Ko)
  retenues_eau.csv  (560.5 Ko)
  va_croissance.csv  (6.4 Ko)
  va_pib.csv  (6.0 Ko)
  va_travailleur.csv  (6.8 Ko)
  va_usd.csv  (6.5 Ko)


## 0. Fonctions utilitaires

In [22]:
def read_csv_smart(path):
    """Lit un CSV en testant encodages et séparateurs automatiquement."""
    for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
        for sep in [',', ';', '\t']:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc)
                if len(df.columns) > 1:
                    return df, enc, sep
            except Exception:
                continue
    raise ValueError(f'Impossible de lire : {path}')


def wkt_to_geodataframe(df, geom_col='geometry', crs='EPSG:4326'):
    """
    Convertit un DataFrame avec une colonne WKT en GeoDataFrame.
    Calcule aussi le centroïde pour les analyses de distance.
    """
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.GeoSeries.from_wkt(df[geom_col]),
        crs=crs
    )
    # Centroïde en WGS84 (utile pour calculs de distance en Phase 4)
    gdf['centroid_lon'] = gdf.geometry.centroid.x
    gdf['centroid_lat'] = gdf.geometry.centroid.y
    return gdf


def check_bounds_togo(gdf, name):
    """Vérifie que les géométries sont dans les bornes du Togo."""
    bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
    ok = (bounds[0] >= -0.2) and (bounds[2] <= 2.0) and (bounds[1] >= 5.5) and (bounds[3] <= 11.5)
    if ok:
        print(f'  ✓ Géométries dans les bornes du Togo')
    else:
        print(f'  ⚠ Bbox [{bounds[0]:.3f}, {bounds[1]:.3f}, {bounds[2]:.3f}, {bounds[3]:.3f}] — à vérifier')


print('Fonctions chargées ✓')

Fonctions chargées ✓


## 1. Sources géospatiales (infrastructures agricoles)

Pour chaque CSV : lecture → détection lat/lon → conversion GeoDataFrame → exploration.

In [23]:
GEO_FILES = {
    'elevage':      RAW / 'elevage.csv',
    'abattoirs':    RAW / 'abattoirs.csv',
    'pisciculture': RAW / 'pisciculture.csv',
    'retenues_eau': RAW / 'retenues_eau.csv',
    'barrages':     RAW / 'barrages.csv',
}

geo_layers  = {}
geo_summary = {}

for name, path in GEO_FILES.items():
    print(f'\n{"="*60}')
    print(f'[{name.upper()}]')

    if not path.exists():
        print(f'  ✗ Fichier introuvable : {path}')
        continue

    try:
        df, enc, sep = read_csv_smart(path)
        print(f'  Lignes × Colonnes : {df.shape}')
        print(f'  Colonnes : {list(df.columns)}')

        # Valeurs manquantes
        null_pct = (df.isnull().sum() / len(df) * 100).round(1)
        non_null = null_pct[null_pct > 0]
        if not non_null.empty:
            print(f'  Valeurs manquantes (%) :')
            print(non_null.to_string())
        else:
            print(f'  Valeurs manquantes : Aucune')

        # Conversion WKT → GeoDataFrame
        gdf = wkt_to_geodataframe(df)
        print(f'  Géométrie : {gdf.geom_type.value_counts().to_dict()}')
        check_bounds_togo(gdf, name)

        # Colonnes admin disponibles
        admin_cols = [c for c in df.columns if 'nom_bdd' in c or 'canton' in c]
        print(f'  Colonnes admin : {admin_cols}')

        geo_layers[name] = gdf
        geo_summary[name] = {
            'n': len(gdf),
            'geom': gdf.geom_type.mode()[0],
            'admin_cols': admin_cols,
            'status': 'GeoDataFrame ✓'
        }

        display(gdf.drop(columns='geometry', errors='ignore').head(3))

    except Exception as e:
        print(f'  ✗ Erreur : {e}')
        import traceback; traceback.print_exc()


[ELEVAGE]
  Lignes × Colonnes : (3004, 14)
  Colonnes : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'nom_localite', 'etab_nom', 'etab_adresse', 'etab_jour', 'etab_creation_date', 'activite_statut', 'activite_categorie', 'toilette_type', 'geometry']
  Valeurs manquantes : Aucune
  Géométrie : {'Polygon': 3004}
  ✓ Géométries dans les bornes du Togo
  Colonnes admin : ['region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,nom_localite,etab_nom,etab_adresse,etab_jour,etab_creation_date,activite_statut,activite_categorie,toilette_type,centroid_lon,centroid_lat
0,_mview_general_etablissements_elevage.fid-54b5...,Centrale,Blitta,Blitta 1,Blitta,Ditcheyidadi,Ditcheyidadi,Tondja -ditcheyidadi,"{Lundi,Mardi,jeudi,Mercredi,Vendredi,Samedi,Di...",1988,{Construit et Utilise},{Elevage traditionnel},{WCs},0.957211,8.280226
1,_mview_general_etablissements_elevage.fid-54b5...,Centrale,Blitta,Blitta 1,Blitta,Tondja,Ferme Panouzi,"Blitta gare,tondja","{Lundi,Mardi,Mercredi,jeudi,Vendredi,Dimanche,...",2012,{Construit et Utilise},{Elevage traditionnel},"{Douches,WCs,Latrines seches}",0.958343,8.279474
2,_mview_general_etablissements_elevage.fid-54b5...,Centrale,Blitta,Blitta 1,Blitta-Village,Tomegbe,Sowougan,Néant,"{Lundi,Mardi,Mercredi,jeudi,Vendredi,Samedi}",2017,{Construit et Utilise},"{Elevage traditionnel,autre}",{autre},1.010151,8.368283



[ABATTOIRS]
  Lignes × Colonnes : (36, 10)
  Colonnes : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'abattoir_nom', 'abattoir_type', 'organisme', 'batiment_fonction', 'geometry']
  Valeurs manquantes : Aucune
  Géométrie : {'Point': 36}
  ✓ Géométries dans les bornes du Togo
  Colonnes admin : ['region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,abattoir_nom,abattoir_type,organisme,batiment_fonction,centroid_lon,centroid_lat
0,_mview_agrianimaux_etablissements_abattoir.fid...,Maritime,Agoè-Nyivé,Agoè-Nyivé 4,Togblekope,Aire d'abattage des petits ruminants d'Agoe-Zongo,Traditionnel,Onaf,"Une parc,un bureau, une garde ,les toilettes",1.207853,6.254587
1,_mview_agrianimaux_etablissements_abattoir.fid...,Maritime,Avé,Avé 1,Tovégan,Togo volaille,autre,Nsp,Nsp,0.920412,6.521438
2,_mview_agrianimaux_etablissements_abattoir.fid...,Maritime,Vo,Vo 1,Vogan,Abattoir de Vogan,Traditionnel,Nsp,Nsp,1.523597,6.340239



[PISCICULTURE]
  Lignes × Colonnes : (141, 9)
  Colonnes : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'etab_adr', 'ouverture_jour', 'organisme', 'geometry']
  Valeurs manquantes : Aucune
  Géométrie : {'MultiPolygon': 141}
  ✓ Géométries dans les bornes du Togo
  Colonnes admin : ['region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,etab_adr,ouverture_jour,organisme,centroid_lon,centroid_lat
0,_mview_general_zones_pisiculture.fid-54b5a690_...,Maritime,Avé,Avé 1,Kévé,Keve vogancope,"{Lundi,Mercredi,Mardi,Jeudi,Vendredi,Samedi,Di...",FAPV,1.032488,6.465044
1,_mview_general_zones_pisiculture.fid-54b5a690_...,Maritime,Avé,Avé 1,Kévé,Keve,"{Lundi,Mardi,Mercredi,Jeudi,Vendredi,Samedi}",LORENOVICH INTERNATIONAL,1.018994,6.463083
2,_mview_general_zones_pisiculture.fid-54b5a690_...,Maritime,Avé,Avé 1,Assahoun,Néant,"{Lundi,Mardi,Mercredi,Jeudi,Vendredi,Samedi,Di...",Nsp,0.908986,6.471479



[RETENUES_EAU]
  Lignes × Colonnes : (611, 10)
  Colonnes : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'village', 'barrage_nom_officiel', 'barrage_type', 'barrage_utilisation', 'geometry']
  Valeurs manquantes : Aucune
  Géométrie : {'MultiPolygon': 611}
  ✓ Géométries dans les bornes du Togo
  Colonnes admin : ['region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,village,barrage_nom_officiel,barrage_type,barrage_utilisation,centroid_lon,centroid_lat
0,_mview_general_retenues_eau_collinaires.fid-54...,Maritime,Avé,Avé 1,Ando,Adekpui,Nsp,Retenue d'eau,{autre},0.805842,6.478082
1,_mview_general_retenues_eau_collinaires.fid-54...,Maritime,Avé,Avé 1,Kévé,Seve-kpota,Nsp,Retenue d'eau,"{elevage,irrigation}",0.962201,6.434850
2,_mview_general_retenues_eau_collinaires.fid-54...,Maritime,Golfe,Golfe 4,Amoutivé,Nyekonakpoé,Barrage Togbato de Nyekonakpoé,Retenue d'eau,"{irrigation,elevage}",1.206113,6.136881



[BARRAGES]
  Lignes × Colonnes : (300, 6)
  Colonnes : ['FID', 'region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd', 'geometry']
  Valeurs manquantes : Aucune
  Géométrie : {'MultiLineString': 300}
  ✓ Géométries dans les bornes du Togo
  Colonnes admin : ['region_nom_bdd', 'prefecture_nom_bdd', 'commune_nom_bdd', 'canton_nom_bdd']


,FID,region_nom_bdd,prefecture_nom_bdd,commune_nom_bdd,canton_nom_bdd,centroid_lon,centroid_lat
0,_mview_general_digues_petits_barrages.fid-54b5...,Maritime,Avé,Avé 2,Badja,1.027890,6.362503
1,_mview_general_digues_petits_barrages.fid-54b5...,Maritime,Avé,Avé 2,Badja,1.007272,6.343431
2,_mview_general_digues_petits_barrages.fid-54b5...,Maritime,Avé,Avé 1,Edzi,0.870935,6.400912


## 2. Sources socio-économiques (séries Banque mondiale)

In [24]:
BM_FILES = {
    'agri_rural':     RAW / 'agri_rural.csv',
    'va_pib':         RAW / 'va_pib.csv',
    'va_croissance':  RAW / 'va_croissance.csv',
    'va_usd':         RAW / 'va_usd.csv',
    'va_travailleur': RAW / 'va_travailleur.csv',
}

bm_frames = {}

for name, path in BM_FILES.items():
    print(f'\n{"-"*55}')
    print(f'[{name.upper()}]')

    if not path.exists():
        print(f'  ✗ Fichier introuvable : {path}')
        continue

    try:
        df, enc, sep = read_csv_smart(path)

        # Nettoyage ligne parasite (ligne 0 = métadonnées HDX type "#country+name")
        if df.iloc[0].astype(str).str.startswith('#').any():
            print(f'  ⚠ Ligne de métadonnées détectée en position 0 — supprimée')
            df = df.iloc[1:].reset_index(drop=True)
            # Reconvertir les types après suppression
            for col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='ignore')

        bm_frames[name] = df
        print(f'  Shape    : {df.shape}')
        print(f'  Colonnes : {list(df.columns)}')

        # Détecter format LONG vs WIDE
        date_cols = [c for c in df.columns if c.lower() in ('year', 'date', 'annee', 'année')]
        fmt = f'LONG — colonne date : {date_cols[0]}' if date_cols else 'WIDE (années en colonnes)' 
        print(f'  Format   : {fmt}')

        # Années disponibles
        if date_cols:
            annees = sorted(df[date_cols[0]].dropna().unique())
            print(f'  Années   : {annees[0]} → {annees[-1]} ({len(annees)} valeurs)')

        display(df.head(4))

    except Exception as e:
        print(f'  ✗ Erreur : {e}')


-------------------------------------------------------
[AGRI_RURAL]
  ⚠ Ligne de métadonnées détectée en position 0 — supprimée
  ✗ Erreur : invalid error value specified

-------------------------------------------------------
[VA_PIB]
  Shape    : (64, 8)
  Colonnes : ['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal']
  Format   : LONG — colonne date : date
  Années   : 1960 → 2023 (64 valeurs)


,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,18.130832,NaN,NaN,1
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,18.700143,NaN,NaN,1
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,18.364991,NaN,NaN,1
3,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2020,19.076887,NaN,NaN,1



-------------------------------------------------------
[VA_CROISSANCE]
  Shape    : (64, 8)
  Colonnes : ['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal']
  Format   : LONG — colonne date : date
  Années   : 1960 → 2023 (64 valeurs)


,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,4.154855,NaN,NaN,1
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,5.103109,NaN,NaN,1
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,3.288445,NaN,NaN,1
3,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2020,3.259057,NaN,NaN,1



-------------------------------------------------------
[VA_USD]
  Shape    : (64, 8)
  Colonnes : ['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal']
  Format   : LONG — colonne date : date
  Années   : 1960 → 2023 (64 valeurs)


,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,1.544291e+09,NaN,NaN,0
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,1.482688e+09,NaN,NaN,0
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,1.410698e+09,NaN,NaN,0
3,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2020,1.365785e+09,NaN,NaN,0



-------------------------------------------------------
[VA_TRAVAILLEUR]
  Shape    : (64, 8)
  Colonnes : ['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal']
  Format   : LONG — colonne date : date
  Années   : 1960 → 2023 (64 valeurs)


,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2023,NaN,NaN,NaN,2
1,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2022,1592.158243,NaN,NaN,2
2,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2021,1543.266447,NaN,NaN,2
3,"Agriculture, forestry, and fishing, value adde...",Togo,TGO,2020,1522.587688,NaN,NaN,2


## 3. Résumé Phase 1

In [25]:
print('=' * 60)
print('RÉSUMÉ PHASE 1')
print('=' * 60)

print('\n[SOURCES GÉOSPATIALES]')
for name, info in geo_summary.items():
    print(f'  {name:15s} | {info["n"]:4d} lignes | {info["status"]}')

print('\n[SÉRIES BANQUE MONDIALE]')
for name, df in bm_frames.items():
    print(f'  {name:15s} | {df.shape[0]:3d} lignes × {df.shape[1]:3d} colonnes')

print('\n[POINTS À CLARIFIER AVANT PHASE 2]')
print('  1. Les colonnes lat/lon sont-elles toutes détectées ? (voir ⚠ ci-dessus)')
print('  2. Quelle colonne contient le nom de la préfecture/région dans chaque source ?')
print('  3. Les séries BM sont-elles en format WIDE ou LONG ?')
print('  4. Y a-t-il des doublons ou des coordonnées aberrantes ?')

RÉSUMÉ PHASE 1

[SOURCES GÉOSPATIALES]
  elevage         | 3004 lignes | GeoDataFrame ✓
  abattoirs       |   36 lignes | GeoDataFrame ✓
  pisciculture    |  141 lignes | GeoDataFrame ✓
  retenues_eau    |  611 lignes | GeoDataFrame ✓
  barrages        |  300 lignes | GeoDataFrame ✓

[SÉRIES BANQUE MONDIALE]
  va_pib          |  64 lignes ×   8 colonnes
  va_croissance   |  64 lignes ×   8 colonnes
  va_usd          |  64 lignes ×   8 colonnes
  va_travailleur  |  64 lignes ×   8 colonnes

[POINTS À CLARIFIER AVANT PHASE 2]
  1. Les colonnes lat/lon sont-elles toutes détectées ? (voir ⚠ ci-dessus)
  2. Quelle colonne contient le nom de la préfecture/région dans chaque source ?
  3. Les séries BM sont-elles en format WIDE ou LONG ?
  4. Y a-t-il des doublons ou des coordonnées aberrantes ?


---
**Prochaine étape** → `02_cleaning.ipynb`  
Compléter `data/processed/data_dictionary.md` avec les colonnes réelles de chaque source.